### Bronze Profile Summary:

Profile Summary script for the Bronze layer has been enhanced with advanced profiling metrics. This script identifies data quality issues such as null distributions and cardinality directly within the audit schema.

In [0]:
import json
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, TimestampType
from datetime import datetime

# --- 1. CONFIGURATION ---
try:
    # Resolve parameters from Job to avoid hard-coding [cite: 28]
    dbutils.widgets.text("datasets_json", "[]", "Datasets JSON Array")
    raw_input = dbutils.widgets.get("datasets_json")
    
    SOURCE_CATALOG = "data_bronze"
    SOURCE_SCHEMA  = "bronze"
    AUDIT_SCHEMA   = "audit"
    AUDIT_TABLE    = f"{SOURCE_CATALOG}.{AUDIT_SCHEMA}.profile_summary"
except Exception as e:
    print(f"Setup failed: {str(e)}")
    raise

# --- 2. MODULAR FUNCTIONS ---

def ensure_audit_table():
    """
    Ensures the audit table exists. 
    NOTE: If you still see the Merge error, drop the table once to reset schema: 
    DROP TABLE IF EXISTS data_bronze.audit.profile_summary
    """
    try:
        if not spark.catalog.tableExists(AUDIT_TABLE):
            print(f"Initializing Audit Table: {AUDIT_TABLE}")
            spark.sql(f"""
                CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
                    dataset_name STRING,
                    layer STRING,
                    row_count LONG,
                    column_count LONG, -- Must be LONG to match Spark default
                    null_count LONG,
                    null_percent DOUBLE,
                    unique_count LONG,
                    columns STRING,
                    profile_timestamp TIMESTAMP
                ) USING DELTA
            """)
            spark.sql(f"COMMENT ON TABLE {AUDIT_TABLE} IS 'Audit log for Bronze layer metrics.'")
    except Exception as e:
        print(f"Critical Error creating audit table: {str(e)}")
        raise

def profile_bronze_table(dataset_name):
    """Calculates metadata, nulls, and uniqueness for a specific Bronze table[cite: 13, 100]."""
    try:
        full_table_name = f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.{dataset_name.lower()}"
        df = spark.table(full_table_name)
        total_rows = df.count()
        num_cols = len(df.columns)
        
        # Calculate nulls across all columns [cite: 51, 100]
        # Null % = (Total Nulls / (Rows * Columns)) * 100
        null_counts_df = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
        total_nulls = sum(null_counts_df.first().asDict().values())
        null_percent = (total_nulls / (total_rows * num_cols)) * 100 if total_rows > 0 else 0.0
        
        # Calculate unique count (Distinct rows)
        unique_count = df.distinct().count()
        
        # Define Metadata Object
        metadata = {
            "dataset_name": dataset_name,
            "layer": "BRONZE",
            "row_count": total_rows,
            "column_count": num_cols,
            "null_count": total_nulls,
            "null_percent": round(null_percent, 2),
            "unique_count": unique_count,
            "columns": ", ".join(df.columns),
            "profile_timestamp": datetime.now()
        }
        
        # Explicit Schema Definition to solve [DELTA_MERGE_INCOMPATIBLE_DATATYPE]
        schema = StructType([
            StructField("dataset_name", StringType(), True),
            StructField("layer", StringType(), True),
            StructField("row_count", LongType(), True),
            StructField("column_count", LongType(), True),
            StructField("null_count", LongType(), True),
            StructField("null_percent", DoubleType(), True),
            StructField("unique_count", LongType(), True),
            StructField("columns", StringType(), True),
            StructField("profile_timestamp", TimestampType(), True)
        ])
        
        # Append logic with specific exception handling
        try:
            spark.createDataFrame([metadata], schema=schema).write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)
            print(f"Profiled: {dataset_name} | Null%: {metadata['null_percent']}% | Uniq: {unique_count}")
        except Exception as write_err:
            print(f"Write failed for {dataset_name}. Potential schema mismatch: {str(write_err)}")
            # Optional: spark.sql(f"ALTER TABLE {AUDIT_TABLE} SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')")
        
    except Exception as e:
        print(f"Profiling calculation failed for {dataset_name}: {str(e)}")

# --- 3. ORCHESTRATION ---

def run_bronze_profiling():
    """Batch executes profiling for the provided dataset array[cite: 91, 159]."""
    try:
        ensure_audit_table()
        dataset_list = json.loads(raw_input)
        
        if not dataset_list:
            print("No datasets found in input parameters.")
            return

        for ds in dataset_list:
            profile_bronze_table(ds)
            
        print(f"\n[SUCCESS] Bronze profiling complete in {AUDIT_TABLE}.")
    except Exception as e:
        print(f"Batch Orchestration failed: {str(e)}")
        raise e

if __name__ == "__main__":
    run_bronze_profiling()

### Unit Testing & Evidence Collection
As per the Deliverable Standards, you must collect evidence of the profiling results.

In [0]:
import json
from pyspark.sql import functions as F

# --- 1. CONFIGURATION & PARAMETERS ---
try:
    # Resolve parameters from Job to avoid hard-coding [cite: 28]
    dbutils.widgets.text("datasets_json", "[]", "Datasets JSON Array")
    raw_input = dbutils.widgets.get("datasets_json")
    
    AUDIT_TABLE = "data_bronze.audit.profile_summary"
except Exception as e:
    print(f"Test Configuration failed: {str(e)}")
    raise

# --- 2. MODULAR TEST FUNCTION ---

def test_profile_metrics_validity(dataset_name):
    """
    Verifies that audit metrics for a specific dataset are within logical bounds.
    [cite: 13, 25, 51]
    """
    try:
        print(f"--- Validating Profiling Metrics: {dataset_name} ---")
        
        # Filter audit table for the specific dataset and the latest profile run [cite: 100]
        audit_df = spark.table(AUDIT_TABLE)
        latest_metrics = (audit_df
                          .filter(F.col("dataset_name") == dataset_name)
                          .orderBy(F.col("profile_timestamp").desc())
                          .limit(1))
        
        if latest_metrics.count() == 0:
            raise AssertionError(f"No profiling data found in {AUDIT_TABLE} for {dataset_name}")

        metrics = latest_metrics.select("null_percent", "row_count", "unique_count").first()
        
        # 1. Null Percentage Boundary Check [cite: 51]
        null_p = metrics['null_percent']
        assert 0 <= null_p <= 100, f"FAILED: Null percent {null_p}% out of bounds for {dataset_name}"
        
        # 2. Row Count Sanity 
        rows = metrics['row_count']
        assert rows >= 0, f"FAILED: Negative row count detected for {dataset_name}"
        
        # 3. Uniqueness Logic Check
        uniq = metrics['unique_count']
        assert uniq <= rows, f"FAILED: Unique count {uniq} exceeds total rows {rows} for {dataset_name}"
        
        print(f"PASSED: {dataset_name} | Null%: {null_p}% | Rows: {rows} | Unique: {uniq}")

    except Exception as e:
        print(f"Profiling Test Failed for {dataset_name}: {str(e)}")
        raise

# --- 3. ORCHESTRATION ---

def run_profiling_unit_tests():
    """Batch executes profiling validations for the provided dataset array."""
    try:
        dataset_list = json.loads(raw_input)
        
        if not dataset_list:
            print("No datasets provided for profiling validation.")
            return

        for ds in dataset_list:
            test_profile_metrics_validity(ds)
            
        print(f"\n[SUCCESS] All profiling metrics verified for Bronze Layer.")

    except Exception as e:
        print(f"Batch Profiling Test Execution Error: {str(e)}")
        raise e

if __name__ == "__main__":
    run_profiling_unit_tests()